In [7]:
import os
import sys

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from ytmusic_library import YTMusicPlaylists, PLAYLIST_LIMIT

import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')

HEADER_FILE='../headers_auth.json'
print(f'Using header file: {HEADER_FILE}')

# quick test for basic library api
RUN_TEST=True
if RUN_TEST:
    from ytmusicapi import YTMusic
    _yt = ytmusicapi.YTMusic(HEADER_FILE)
    assert(_yt)
    assert(_yt.get_song('Kv7K9ghgcgA')) # Don't Think Twice, It's All Right	Bob Dylan
    assert(_yt.get_library_playlists(limit=1))
    assert(_yt.get_library_albums())
    assert(_yt.get_library_artists())
    del _yt
    del YTMusic

Using ytmusicapi version: 0.24.0
Using header file: ../headers_auth.json


In [2]:
Y = YTMusicPlaylists(header=HEADER_FILE)
print(Y.playlists['title'].unique())

['Your Likes' 'Acoustic Guitar Explorations like' 'ambiant electro'
 'ambient' 'ambient BOC' 'ambient classic'
 'ambient Dream Pop Deep Sleep like' 'ambient haunting harmonious like'
 'ambient Indie synths' 'ambient Indie Synths radio' 'ambient lynchian'
 'ambient modern' 'ambient piano' 'Ambient Psychill' 'beats'
 'beats cosmic Slop like' 'Beats indie Chill like'
 'Beats indie Chill radio' 'beats instrumental' 'Beats Lofi Loft like'
 'beats radio' 'beats Soulful Instrumentals' 'beats trap dj'
 'Beats Without Rhymes like' "beats wonky LA '10 scene" 'beats_chill dj'
 'beats_jazzy dj' 'beats_lofi' 'beats_phat dj' 'beats_raw dj'
 'beats_soul dj' 'beats_wonky dj' 'bluegrass billy' 'blues'
 'blues chicago' 'blues delta radio' 'blues delta roots like'
 'blues radio' 'blues texas roots radio like' 'Bossa Nova like'
 'Bossa Nova radio' 'brass' 'Brass n chill' 'Chill Supermix' 'Chillwave'
 'doo wop like' 'doo wop radio' 'electronic 2000s like'
 'electronic 2000s radio' 'Electronic 2010s like'
 

In [3]:
res = Y.yt.search(query='Hot Fuss', filter='albums', limit=3)

## Split mixed rating playlist into LIKE and INDIFFERENT


In [8]:
# For each playlist, Create Unrated and Liked Subset Playlist, delete original
playlist_names = [

]

# History
"""
'Rock 1980s New Wave radio', 'x_r.60sMusic_tracks_radio', 'x_r.70sMusic_tracks_radio'
'x_r.treemusic radio', 'x_r.nudisco_tracks_radio', 'x_r.Rock_tracks_radio', 
'x_r.hiphop_tracks_radio', 'x_r.futurebeats_tracks_radio', 'x_r.indie_tracks_radio',
'x_r.blues_tracks_radio', 'x_r.futurebass_tracks_radio','soul radio', 'Reggae radio', 
'trip hop radio', 'psychedelic classic rock radio', 'psych rock radio', 'Acoustic Guitar Explorations radio',
'x_r.90shiphop_tracks_radio', 'Hip Hop 2000s radio', 'Produced by dilla', 
'electronic indie radio', 'jazz cool radio', 'jazz radio', 'x_r.jazznoir_tracks_radio',
'Electronic House Special radio', 'electronic 2000s radio', 'Shoegaze radio', 'blues radio', 'blues delta radio', 'beats radio', 
'Lofi Loft indifferent', 'Lofi House', 'Deep Minimum', 'Soulful House Vibe', Lofi/Hifi house', 'studio beats', 'House', 'lo fi',
'rock classic radio' 'Shoegaze radio', 'x_r.90shiphop_tracks_radio', 'x_r.triphop_tracks_radio', 
'x_r.GypsyJazz_tracks_radio', 'x_r.Gfunk_tracks_radio',  'summertime covers', 
'x_r.AfricanMusic_tracks_radio', 'x_r.jazznoir_tracks_radio', 'jazzyhiphop_tracks_radio', 'x_r.jazz_tracks_radio', 
'jazz solo guitar radio',  'jazz radio', 'Jazz Feels the Blues', 
'jazz cool radio', 'rock classic radio',  'electronic indie radio', 'rock stoner sludge dank radio' ,
'Hip Hop Classic West Coast radio', 'trip hop radio', 'x_r.ClassicRock_tracks_radio'
"""

for playlist_name in playlist_names:
    Y.create_like_and_unrated_rating_playlist_subset(playlist_name, verbose=True)

## Rate playlists that should have all LIKE as LIKE

In [ ]:
MAX_PLAYLIST_SIZE_TO_RATE = 6000
for title, playlistId in Y.playlist_get_all_like_playlists().items():
    try:
        print(100*'=')
        info = Y.playlist_get_info(playlistId, playlist_limit=MAX_PLAYLIST_SIZE_TO_RATE)
        num_tracks = len(info['tracks'])
        if num_tracks >= MAX_PLAYLIST_SIZE_TO_RATE:
            print(f'Skipping playlist: {title} ({playlistId}) which has {MAX_PLAYLIST_SIZE_TO_RATE} or more tracks')
            continue
        print(f'Playlist: {title} ({playlistId}) has {num_tracks} tracks')
        Y.playlist_rate_all_songs(playlistId, rating='LIKE')
    except Exception as e:
        print(e)

## Remove duplicate entries from playlists

In [49]:
import pandas as pd
pl_cache = {}
fixed = []
DUPLICATE_THRESHOLD=4

for i, p_row in Y.playlists.iterrows():
    if i == 0: # skip auto likes playlist
        continue

    pid = p_row.playlistId
    if pid in pl_cache:
        pl =  pl_cache[pid]
    else:
        pl = Y.yt.get_playlist(playlistId=pid, limit=PLAYLIST_LIMIT)
        pl_cache[pid] = pl
        
    tracks = [t['videoId'] for t in pl['tracks']]
    tracks_unique = pd.Series(tracks).unique().tolist()
    n_dupes = len(tracks)-len(tracks_unique)
    if n_dupes >= DUPLICATE_THRESHOLD:
        print(f"{n_dupes} dupe tracks will be removed from playlist: {p_row.title} ({len(tracks_unique)} of {len(tracks)} unique)")
        if pid not in fixed:
            Y.yt.create_playlist(title=str(pl['title']), description=str(pl['description']), video_ids=tracks_unique)
            Y.yt.delete_playlist(playlistId=pid)
            fixed.append(pid)


4 dupe tracks will be removed from playlist: ambient classic (49 of 53 unique)
47 dupe tracks will be removed from playlist: beats (708 of 755 unique)
8 dupe tracks will be removed from playlist: beats instrumental (266 of 274 unique)
15 dupe tracks will be removed from playlist: Brass n chill (162 of 177 unique)
6 dupe tracks will be removed from playlist: Chillwave (358 of 364 unique)
21 dupe tracks will be removed from playlist: Electronic House Special like (116 of 137 unique)
13 dupe tracks will be removed from playlist: Folk like (91 of 104 unique)
47 dupe tracks will be removed from playlist: future beats (688 of 735 unique)
6 dupe tracks will be removed from playlist: garage rock (308 of 314 unique)
12 dupe tracks will be removed from playlist: goth 1980s radio (129 of 141 unique)
72 dupe tracks will be removed from playlist: Hip Hop 1990s like (509 of 581 unique)
92 dupe tracks will be removed from playlist: Hip Hop Hits liked (188 of 280 unique)
1339 dupe tracks will be remov

## Get Public Playlists

In [14]:
Y.get_playlists_by_privacy(privacy='PUBLIC')

Found public playlist named: Your Likes
Found public playlist named: Jazz Guitar
Found public playlist named: Kurt Cobain's Record Collection
Found public playlist named: The World of Elliott Smith
Found public playlist named: Inspired by True Detective
Found public playlist named: Jazz Feels the Blues
Found public playlist named: Traditional Jazz
Found public playlist named: jazz Ted Gioia’s How to Listen to Jazz
Found public playlist named: Chill Supermix
Found public playlist named: The World of Lou Reed and the Velvet Underground
Found public playlist named: Vintage Christmas Crooners


title                                                 Your Likes
playlistId                                                    LM
thumbnails     [{'url': 'https://www.gstatic.com/youtube/medi...
description                                        Auto playlist
count                                                        NaN
                                     ...                        
playlistId           RDCLAK5uy_lu_DWkCX9Qz5upUBcvrs3nv8dgpbwxc0w
thumbnails     [{'url': 'https://lh3.googleusercontent.com/vx...
description                             YouTube Music • 59 songs
count                                                         59
author                   [{'name': 'YouTube Music', 'id': None}]
Length: 66, dtype: object